# Notebook 06: Phase Lock Demo — Interactive 45° Boundary Explorer

## Objectives

1. **Interactive exploration** of the 45° critical boundary
2. **Real-time visualization** of how angle affects stability
3. **Drag-and-test** different claim types (math, equations, embeddings, agents)
4. **Phase transition simulation** — watch a claim drift from 0° to 90°
5. **Synthesis notebook** — ties together all previous notebooks

## Core Principle

> **The 45° boundary is where VC (Valid Construction) collapses into IA (Invalid Assignment).**
> 
> This notebook lets you *feel* that transition.

## Reference

This notebook integrates concepts from:
- `00_angle_theory.ipynb` — Geometric foundation
- `02_math_claims_hydration.ipynb` — PCSP claims
- `03_equation_consistency.ipynb` — Equation systems
- `04_ml_embedding_alignment.ipynb` — Embedding drift
- `05_agent_consistency_check.ipynb` — Agent alignment

## 1. Setup and Core Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Wedge
import sys
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')

# Colab setup
if 'google.colab' in sys.modules:
    !pip install numpy matplotlib ipywidgets > /dev/null 2>&1
    from IPython.display import display, HTML, clear_output
    display(HTML("""
    <style>
    .widget-label { font-weight: bold; }
    .output_text { font-family: monospace; }
    </style>
    """))
    print("✓ Colab environment ready")
else:
    print("✓ Local environment ready")

# Core functions (consolidated from all notebooks)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def angle_degrees(a: np.ndarray, b: np.ndarray) -> float:
    cos = cosine_similarity(a, b)
    return np.arccos(np.clip(cos, -1.0, 1.0)) * 180 / np.pi

def lock_status(angle: float) -> Dict[str, object]:
    if angle < 10:
        return {"status": "VC / GOS", "phase": "LOCKED", "interpretation": "Valid Construction — stable", 
                "color": "#2ecc71", "signal_noise": "Signal >> Noise", "emoji": "🔒✅"}
    elif angle < 35:
        return {"status": "weak VC", "phase": "DRIFTING", "interpretation": "Partially constructed — monitor", 
                "color": "#f1c40f", "signal_noise": "Signal > Noise", "emoji": "⚠️"}
    elif angle <= 55:
        return {"status": "IA / ZOS", "phase": "CRITICAL (45°)", "interpretation": "Invalid Assignment — collapsing", 
                "color": "#e74c3c", "signal_noise": "Signal ≈ Noise", "emoji": "🔴💀"}
    elif angle < 80:
        return {"status": "weak IA", "phase": "DECOUPLED", "interpretation": "Mostly unconstrained — unreliable", 
                "color": "#e67e22", "signal_noise": "Noise > Signal", "emoji": "⚠️💀"}
    else:
        return {"status": "orthogonal", "phase": "BROKEN", "interpretation": "No relationship — discard", 
                "color": "#95a5a6", "signal_noise": "Noise Only", "emoji": "⚰️"}

print("✓ Core functions loaded")
print("\n" + "=" * 50)
print("PHASE LOCK DEMO — Interactive 45° Boundary Explorer")
print("=" * 50)

## 2. Interactive Angle Explorer

Drag the slider to see how stability changes as angle increases from 0° to 90°.

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider, VBox, HBox, Label, HTML as WHTML
from IPython.display import display

def update_angle_demo(angle_deg: float):
    """Update the angle visualization in real-time."""
    clear_output(wait=True)
    
    # Get status
    status = lock_status(angle_deg)
    cos_val = np.cos(np.radians(angle_deg))
    
    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # === Plot 1: Cosine curve with current position ===
    ax1 = axes[0, 0]
    angles = np.linspace(0, 90, 200)
    cos_vals = np.cos(np.radians(angles))
    ax1.plot(angles, cos_vals, 'k-', linewidth=2.5)
    ax1.fill_between(angles, 0, cos_vals, where=(angles >= angle_deg-2) & (angles <= angle_deg+2), 
                      color=status['color'], alpha=0.5)
    ax1.scatter([angle_deg], [cos_val], s=200, c=status['color'], edgecolors='black', zorder=5, marker='D')
    ax1.axvline(10, color='#2ecc71', linestyle='--', alpha=0.5, label='VC Lock (10°)')
    ax1.axvline(45, color='#e74c3c', linestyle='--', alpha=0.5, linewidth=2, label='Critical (45°)')
    ax1.axvline(35, color='#f1c40f', linestyle=':', alpha=0.5)
    ax1.axvline(55, color='#f1c40f', linestyle=':', alpha=0.5)
    ax1.set_xlim(0, 90)
    ax1.set_ylim(0, 1)
    ax1.set_xlabel('Angle (degrees)', fontsize=10)
    ax1.set_ylabel('cos(θ) — stability', fontsize=10)
    ax1.set_title('Cosine Constraint: Position on Curve', fontweight='bold')
    ax1.legend(loc='upper right', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # === Plot 2: Gauge meter ===
    ax2 = axes[0, 1]
    # Create a semicircle gauge
    theta = np.linspace(0, np.pi, 100)
    r = 1
    x_gauge = r * np.cos(theta)
    y_gauge = r * np.sin(theta)
    ax2.plot(x_gauge, y_gauge, 'k-', linewidth=2)
    
    # Color zones
    zone_boundaries = [0, 10, 35, 55, 90]
    zone_colors = ['#2ecc71', '#f1c40f', '#e74c3c', '#e67e22', '#95a5a6']
    
    for i in range(len(zone_boundaries)-1):
        start_angle = np.radians(90 - zone_boundaries[i+1])
        end_angle = np.radians(90 - zone_boundaries[i])
        ax2.add_patch(Wedge((0, 0), 0.8, 
                      np.degrees(start_angle), np.degrees(end_angle), 
                      facecolor=zone_colors[i], alpha=0.3))
    
    # Needle
    needle_angle = np.radians(90 - angle_deg)
    needle_x = [0, 0.7 * np.cos(needle_angle)]
    needle_y = [0, 0.7 * np.sin(needle_angle)]
    ax2.plot(needle_x, needle_y, 'k-', linewidth=2, color='black')
    ax2.scatter([0], [0], s=100, c='black', zorder=5)
    
    ax2.set_xlim(-1.2, 1.2)
    ax2.set_ylim(-0.2, 1.2)
    ax2.set_aspect('equal')
    ax2.axis('off')
    ax2.set_title('Stability Gauge', fontweight='bold')
    
    # Add angle text
    ax2.text(0, -0.15, f'{angle_deg:.1f}°', ha='center', fontsize=16, fontweight='bold', color=status['color'])
    
    # === Plot 3: Phase transition diagram ===
    ax3 = axes[1, 0]
    
    # Draw the phase boundary at 45°
    ax3.add_patch(FancyBboxPatch((0, 0.4), 45, 0.2, boxstyle="round,pad=0.02", 
                                   facecolor='#2ecc71', alpha=0.5, edgecolor='black'))
    ax3.add_patch(FancyBboxPatch((45, 0.4), 45, 0.2, boxstyle="round,pad=0.02", 
                                   facecolor='#e74c3c', alpha=0.5, edgecolor='black'))
    
    # Arrow showing current position
    arrow_x = angle_deg
    ax3.annotate('', xy=(arrow_x, 0.6), xytext=(arrow_x, 0.9),
                 arrowprops=dict(arrowstyle='->', color=status['color'], lw=2))
    
    ax3.set_xlim(0, 90)
    ax3.set_ylim(0, 1)
    ax3.set_xlabel('Angle (degrees)', fontsize=10)
    ax3.set_yticks([])
    ax3.set_title('Phase Boundary: VC (Green) ← 45° → IA (Red)', fontweight='bold')
    ax3.text(22, 0.45, 'CONSTRUCTION\n(VC/GOS)', ha='center', fontsize=9, fontweight='bold', color='#2ecc71')
    ax3.text(67, 0.45, 'ASSIGNMENT\n(IA/ZOS)', ha='center', fontsize=9, fontweight='bold', color='#e74c3c')
    ax3.axvline(45, color='black', linestyle='-', linewidth=2)
    ax3.text(45, 0.85, 'CRITICAL\nBOUNDARY', ha='center', fontsize=8, fontweight='bold', color='#e74c3c')
    
    # === Plot 4: Status card ===
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Create a status card
    status_text = f"""
    ╔════════════════════════════════════════╗
    ║           CONSISTENCY STATUS           ║
    ╠════════════════════════════════════════╣
    ║  Angle:        {angle_deg:>6.1f}°                    ║
    ║  Status:       {status['status']:>20} ║
    ║  Phase:        {status['phase']:>20} ║
    ║  Signal/Noise: {status['signal_noise']:>20} ║
    ║  cos(θ):       {cos_val:>8.4f}                 ║
    ╠════════════════════════════════════════╣
    ║  {status['emoji']}  {status['interpretation']}  ║
    ╚════════════════════════════════════════╝
    """
    ax4.text(0.5, 0.5, status_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='center', horizontalalignment='center',
             fontfamily='monospace', bbox=dict(boxstyle='round', facecolor=status['color'], alpha=0.2))
    
    plt.suptitle(f'🔬 Phase Lock Demo — {status["emoji"]} {status["status"]} at {angle_deg:.1f}°', 
                 fontsize=14, fontweight='bold', color=status['color'])
    plt.tight_layout()
    plt.show()
    
    # Print additional info
    if 35 <= angle_deg <= 55:
        print(f"\n{'='*50}")
        print(f"🔴 CRITICAL ZONE: Angle = {angle_deg:.1f}°")
        print(f"{'='*50}")
        print("You are in the 45° CRITICAL ZONE.")
        print("At this angle, the claim is neither constructed nor assigned.")
        print("It is maximally unstable and will COLLAPSE into inconsistency.")
        print("This matches the IA/ZOS path in triplet-phase-lock.")
    elif angle_deg < 10:
        print(f"\n{'='*50}")
        print(f"✅ LOCKED ZONE: Angle = {angle_deg:.1f}°")
        print(f"{'='*50}")
        print("You are in the VC/GOS LOCKED ZONE.")
        print("The claim respects all constraints and is STABLE.")
        print("Signal dominates noise — consistent construction.")

# Create the interactive widget
print("\n" + "=" * 50)
print("INTERACTIVE ANGLE EXPLORER")
print("=" * 50)
print("\nDrag the slider to change the angle from 0° to 90°.")
print("Watch how the stability changes as you cross 10° and 45°.\n")

interact(update_angle_demo,
         angle_deg=FloatSlider(min=0, max=90, step=1, value=0, 
                               description='Angle (°):', 
                               style={'description_width': 'initial'},
                               layout={'width': '500px'}))

## 3. Claim Type Explorer

Test different types of claims and see where they fall on the angle spectrum.

In [ ]:
# Define example claims from each domain
example_claims = {
    "Math (PCSP)": {
        "Valid (VC)": {
            "text": "'Some dichotomy results for Mal'cev algebras; general case open'",
            "vector": [1.0, 0.8, 0.9, 0.9, 1.0],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        },
        "Invalid (IA)": {
            "text": "'The PCSP dichotomy is true for all finite algebras'",
            "vector": [0.0, 0.0, 0.0, 0.7, 0.0],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        }
    },
    "Equation": {
        "Valid (VC)": {
            "text": "'2 = 1 + 1' → '24/25 = 0.96' → '√-1.96 = 1.4i'",
            "vector": [1.0, 1.0, 1.0, 0.9, 1.0],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        },
        "Invalid (IA)": {
            "text": "'(1-1=0)' → '(0↦0.96)' → '(2=0.96)'",
            "vector": [0.2, 0.1, 0.0, 0.1, 0.2],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        }
    },
    "ML Embedding": {
        "Valid (VC)": {
            "text": "Well-trained BERT embedding (in-distribution)",
            "vector": [0.95, 0.92, 0.90, 0.88, 0.91],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        },
        "Invalid (IA)": {
            "text": "Drifted embedding (out-of-distribution, 45° critical)",
            "vector": [0.45, 0.50, 0.40, 0.55, 0.48],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        }
    },
    "Agent": {
        "Valid (VC)": {
            "text": "'I will act in user's interest' → actions match",
            "vector": [0.95, 0.92, 0.98, 0.90, 0.94],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        },
        "Invalid (IA)": {
            "text": "'I will act in user's interest' → actions contradict",
            "vector": [0.45, 0.50, 0.40, 0.55, 0.48],
            "constraint_basis": [1.0, 1.0, 1.0, 1.0, 1.0]
        }
    }
}

def angle_from_vector(vec, basis):
    return angle_degrees(np.array(vec), np.array(basis))

print("=" * 60)
print("CLAIM TYPE EXPLORER")
print("=" * 60)
print("\nHere are example claims from each domain and their angles:\n")

for domain, types in example_claims.items():
    print(f"\n{domain}:")
    for validity, data in types.items():
        angle = angle_from_vector(data['vector'], data['constraint_basis'])
        status = lock_status(angle)
        print(f"  {validity:12} ({angle:5.1f}°) {status['emoji']}: {data['text'][:60]}...")

## 4. Interactive Claim Tester

Test your own claim by adjusting constraint adherence scores.

In [ ]:
from ipywidgets import Textarea

def test_custom_claim(c1=0.5, c2=0.5, c3=0.5, c4=0.5, c5=0.5, claim_text=""):
    vec = np.array([c1, c2, c3, c4, c5])
    basis = np.array([1.0, 1.0, 1.0, 1.0, 1.0])
    angle = angle_degrees(vec, basis)
    status = lock_status(angle)
    
    clear_output(wait=True)
    
    print("=" * 60)
    print("CUSTOM CLAIM TESTER")
    print("=" * 60)
    if claim_text:
        print(f"\nClaim: {claim_text}")
    print(f"\nConstraint adherence scores:")
    print(f"  C1 (Anchored to known results): {c1}")
    print(f"  C2 (Specific domain): {c2}")
    print(f"  C3 (Constraints defined): {c3}")
    print(f"  C4 (Construction/proof): {c4}")
    print(f"  C5 (Boundaries acknowledged): {c5}")
    print(f"\nAngle from constraint basis: {angle:.1f}°")
    print(f"Status: {status['status']} {status['emoji']}")
    print(f"Phase: {status['phase']}")
    print(f"Signal/Noise: {status['signal_noise']}")
    print(f"Interpretation: {status['interpretation']}")
    
    # Quick visual
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    
    # Bar chart
    categories = ['C1', 'C2', 'C3', 'C4', 'C5']
    values = [c1, c2, c3, c4, c5]
    colors_bar = [status['color'] if v > 0.7 else '#e74c3c' if v < 0.3 else '#f1c40f' for v in values]
    ax1.bar(categories, values, color=colors_bar, alpha=0.7, edgecolor='black')
    ax1.set_ylim(0, 1)
    ax1.set_ylabel('Constraint Respect')
    ax1.set_title('Constraint Profile')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Position on spectrum
    angles_plot = np.linspace(0, 90, 100)
    stability = np.cos(np.radians(angles_plot))
    ax2.plot(angles_plot, stability, 'k-', linewidth=2)
    ax2.axvline(angle, color=status['color'], linestyle='--', linewidth=2, label=f'Your claim: {angle:.1f}°')
    ax2.axvline(45, color='red', linestyle=':', alpha=0.5)
    ax2.axvline(10, color='green', linestyle=':', alpha=0.5)
    ax2.fill_between(angles_plot, 0, stability, where=(angles_plot > 35) & (angles_plot < 55), 
                      color='red', alpha=0.1)
    ax2.fill_between(angles_plot, 0, stability, where=angles_plot < 10, 
                      color='green', alpha=0.1)
    ax2.set_xlabel('Angle (degrees)')
    ax2.set_ylabel('cos(θ)')
    ax2.set_title('Position on Stability Spectrum')
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 50)
print("TEST YOUR OWN CLAIM")
print("=" * 50)
print("\nAdjust the sliders to create your claim profile.")
print("1.0 = fully respects constraint, 0.0 = ignores constraint\n")

interact(test_custom_claim,
         claim_text=Textarea(value="", placeholder="Describe your claim here...", description="Claim:"),
         c1=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='C1: Anchored'),
         c2=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='C2: Specific'),
         c3=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='C3: Constrained'),
         c4=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='C4: Constructed'),
         c5=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='C5: Boundaries'))

## 5. Phase Transition Simulation

Watch a claim drift from 0° (stable VC) to 90° (broken) and see the collapse at 45°.

In [ ]:
from ipywidgets import Button
import time

def run_phase_transition():
    """Animate the phase transition from 0° to 90°."""
    clear_output(wait=True)
    
    print("=" * 60)
    print("PHASE TRANSITION SIMULATION")
    print("=" * 60)
    print("\nWatching a claim drift from 0° (stable) to 90° (broken)...")
    print("Pay attention to what happens at 45°:\n")
    
    for angle in range(0, 91, 3):
        clear_output(wait=True)
        status = lock_status(angle)
        
        # Create a simple visual
        fig, ax = plt.subplots(figsize=(10, 4))
        
        # Progress bar
        ax.barh([0], [angle], color=status['color'], height=0.3, edgecolor='black')
        ax.axvline(10, color='green', linestyle='--', alpha=0.7, label='VC Lock (10°)')
        ax.axvline(35, color='orange', linestyle=':', alpha=0.7, label='Warning (35°)')
        ax.axvline(45, color='red', linestyle='--', alpha=0.7, linewidth=2, label='CRITICAL (45°)')
        ax.axvline(55, color='orange', linestyle=':', alpha=0.7)
        ax.set_xlim(0, 90)
        ax.set_xlabel('Angle (degrees)')
        ax.set_yticks([])
        ax.set_title(f'Phase Transition: {angle}° — {status["status"]} {status["emoji"]}')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3, axis='x')
        
        # Add status text
        ax.text(angle, -0.15, f'{angle}°', ha='center', fontsize=10, fontweight='bold', color=status['color'])
        
        plt.tight_layout()
        display(plt.gcf())
        
        # Print critical warning at 45°
        if angle == 45:
            print("\n" + "🔴" * 20)
            print("🔴 CRITICAL: The claim has reached 45°!")
            print("🔴 This is the collapse point — IA/ZOS inconsistency.")
            print("🔴" * 20 + "\n")
        
        time.sleep(0.08)
        plt.close()
    
    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print("=" * 60)
    print("\nKey observation: The claim was stable until 10°,")
    print("entered the danger zone after 35°, and COLLAPSED at 45°.")
    print("\nThis matches the triplet-phase-lock framework:")
    print("  Π⁽⁰⁾ expand (0°-10°) → stable")
    print("  Π⁽¹⁾ extend (10°-35°) → drifting")
    print("  Π⁽²⁾ resist (35°-55°) → critical, collapse")
    print("  Π⁽³⁾ synthesis → requires return to <10°")

# Button to run simulation
run_button = Button(description="▶ Run Phase Transition Simulation", button_style='primary')
run_button.on_click(lambda b: run_phase_transition())
display(run_button)
print("\nClick the button above to watch a claim drift from 0° to 90°.")

## 6. Triplet Progression Visualizer

Map the four triplet phases (Π⁽⁰⁾ to Π⁽³⁾) to angles on the cosine curve.

In [ ]:
def show_triplet_progression():
    """Visualize the triplet progression phases on the cosine curve."""
    clear_output(wait=True)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    angles = np.linspace(0, 90, 200)
    stability = np.cos(np.radians(angles))
    ax.plot(angles, stability, 'k-', linewidth=3, label='cos(θ)')
    
    # Define phase regions
    phases = [
        ("Π⁽⁰⁾\nexpand", 0, 10, "#2ecc71", "Stable expansion"),
        ("Π⁽¹⁾\nextend", 10, 35, "#f1c40f", "Drifting extension"),
        ("Π⁽²⁾\nresist", 35, 55, "#e74c3c", "Critical — collapse"),
        ("Π⁽³⁾\nsynthesis", 55, 90, "#95a5a6", "Broken — needs rebuild")
    ]
    
    for name, start, end, color, desc in phases:
        ax.axvspan(start, end, alpha=0.15, color=color)
        mid = (start + end) / 2
        ax.text(mid, 0.2, name, ha='center', fontsize=11, fontweight='bold', color=color)
        ax.text(mid, 0.1, desc, ha='center', fontsize=8, alpha=0.7)
    
    # Mark critical boundary
    ax.axvline(45, color='#e74c3c', linestyle='--', linewidth=2.5, alpha=0.8, label='45° Critical Boundary')
    ax.axvline(10, color='#2ecc71', linestyle='--', linewidth=1.5, alpha=0.5, label='VC Lock Boundary (10°)')
    
    # Annotations
    ax.annotate('Valid Construction\n(VC/GOS)', xy=(5, 0.95), xytext=(5, 0.75),
                ha='center', fontsize=10, fontweight='bold', color='#2ecc71',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.annotate('Invalid Assignment\n(IA/ZOS)', xy=(50, 0.65), xytext=(50, 0.45),
                ha='center', fontsize=10, fontweight='bold', color='#e74c3c',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Add example points
    examples = [
        (5, "VC Claim\n(PCSP results)", "#2ecc71"),
        (20, "Weak VC\n(Slight drift)", "#f1c40f"),
        (45, "IA Claim\n(2 = 0.96)", "#e74c3c"),
        (70, "Weak IA\n(Noisy embedding)", "#e67e22"),
        (88, "Broken\n(Random)", "#95a5a6")
    ]
    
    for angle, label, color in examples:
        cos_val = np.cos(np.radians(angle))
        ax.scatter(angle, cos_val, s=150, c=color, edgecolors='black', zorder=5, marker='D')
        ax.annotate(label, xy=(angle, cos_val), xytext=(angle + 3, cos_val - 0.1),
                    fontsize=7, ha='left', bbox=dict(boxstyle='round', facecolor=color, alpha=0.2))
    
    ax.set_xlim(0, 90)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Angle (degrees)', fontsize=12)
    ax.set_ylabel('cos(θ) — stability', fontsize=12)
    ax.set_title('Triplet Phase-Lock Progression: Π⁽⁰⁾ → Π⁽¹⁾ → Π⁽²⁾ → Π⁽³⁾', 
                 fontsize=14, fontweight='bold')
    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print explanation
    print("\n" + "=" * 60)
    print("TRIPLET PROGRESSION EXPLANATION")
    print("=" * 60)
    print("""
    Π⁽⁰⁾ expand (0°-10°)  →  LOCKED ZONE
        The claim expands within constraints. Stable construction.
        
    Π⁽¹⁾ extend (10°-35°) →  DRIFTING ZONE
        The claim extends beyond core constraints. Monitor for collapse.
        
    Π⁽²⁾ resist (35°-55°) →  CRITICAL ZONE (45° boundary)
        The claim resists constraints → COLLAPSE at 45°.
        This is the IA/ZOS path: collapse → assignment → denial → inconsistency.
        
    Π⁽³⁾ synthesis (>55°)   →  BROKEN ZONE
        The claim is decoupled from constraints. Requires complete rebuild.
    """)

show_triplet_progression()

## 7. Summary Dashboard (Fixed)

A comprehensive dashboard showing all key information.

In [ ]:
def show_dashboard():
    """Display a comprehensive summary dashboard."""
    clear_output(wait=True)
    
    fig = plt.figure(figsize=(14, 12))
    
    # Title
    fig.suptitle('📊 Cosine Constraint Lab — Summary Dashboard', fontsize=16, fontweight='bold', y=0.98)
    
    # === Panel 1: Angle Zones ===
    ax1 = plt.subplot(2, 2, 1)
    angles = np.linspace(0, 90, 100)
    stability = np.cos(np.radians(angles))
    ax1.plot(angles, stability, 'k-', linewidth=2)
    
    zones = [
        (0, 10, '#2ecc71', 'VC Locked'),
        (10, 35, '#f1c40f', 'Weak VC'),
        (35, 55, '#e74c3c', 'CRITICAL'),
        (55, 80, '#e67e22', 'Weak IA'),
        (80, 90, '#95a5a6', 'Broken')
    ]
    
    for start, end, color, label in zones:
        ax1.axvspan(start, end, alpha=0.2, color=color)
        mid = (start + end) / 2
        ax1.text(mid, 0.1, label, ha='center', fontsize=8, fontweight='bold', color=color)
    
    ax1.axvline(45, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax1.set_xlim(0, 90)
    ax1.set_ylim(0, 1)
    ax1.set_xlabel('Angle (degrees)')
    ax1.set_ylabel('cos(θ)')
    ax1.set_title('1. Angle Zones', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # === Panel 2: Status Table (Fixed) ===
    ax2 = plt.subplot(2, 2, 2)
    ax2.axis('off')
    
    table_data = [
        ['Angle', 'Status', 'Phase', 'Signal:Noise'],
        ['< 10°', 'VC / GOS', 'LOCKED', 'Signal >> Noise'],
        ['10-35°', 'weak VC', 'DRIFTING', 'Signal > Noise'],
        ['35-55°', 'IA / ZOS', 'CRITICAL', 'Signal ≈ Noise'],
        ['55-80°', 'weak IA', 'DECOUPLED', 'Noise > Signal'],
        ['> 80°', 'orthogonal', 'BROKEN', 'Noise Only']
    ]
    
    # Create table with correct dimensions
    table = ax2.table(cellText=table_data, loc='center', cellLoc='center', colWidths=[0.2, 0.25, 0.25, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)
    
    # Color header row (row 0, all columns)
    for col in range(4):  # 4 columns
        table[(0, col)].set_facecolor('#34495e')
        table[(0, col)].set_text_props(weight='bold', color='white')
    
    ax2.set_title('2. Status Lookup Table', fontweight='bold')
    
    # === Panel 3: Key Insights ===
    ax3 = plt.subplot(2, 2, 3)
    ax3.axis('off')
    insights = [
        "🔑 The 45° boundary separates construction from assignment.",
        "🔑 VC/GOS = constraint-preserving construction.",
        "🔑 IA/ZOS = unconstrained assignment → collapse.",
        "🔑 Signal > Noise when angle < 10°.",
        "🔑 At 45°, noise equals signal → maximum instability.",
        "🔑 Triplet progression: expand → extend → resist → synthesis."
    ]
    
    y_pos = 0.9
    for insight in insights:
        ax3.text(0.05, y_pos, insight, transform=ax3.transAxes, fontsize=9, verticalalignment='top')
        y_pos -= 0.12
    ax3.set_title('3. Key Insights', fontweight='bold')
    
    # === Panel 4: Formula ===
    ax4 = plt.subplot(2, 2, 4)
    ax4.axis('off')
    
    formula_text = """
    The Cosine Constraint:
    
         cos(θ) = (C · B) / (|C| |B|)
    
    where:
        C = claim vector
        B = constraint basis
        θ = angular distance
    
    Stability = cos(θ)
    
    Critical boundary: θ = 45°
    cos(45°) = √2/2 ≈ 0.707
    
    At θ = 45°, the claim is maximally
    unstable → collapse into IA/ZOS.
    """
    
    ax4.text(0.05, 0.95, formula_text, transform=ax4.transAxes, fontsize=9, 
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))
    ax4.set_title('4. Mathematical Foundation', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

show_dashboard()

## 8. Conclusion

### What You've Learned

| Concept | Summary |
|---------|---------|
| **The 45° Boundary** | The critical threshold where Valid Construction collapses into Invalid Assignment |
| **VC / GOS** | Constraint-preserving construction — angle < 10°, signal dominates |
| **IA / ZOS** | Unconstrained assignment — angle ≈ 45°, noise equals signal, collapse occurs |
| **Triplet Progression** | Π⁽⁰⁾ expand → Π⁽¹⁾ extend → Π⁽²⁾ resist → Π⁽³⁾ synthesis |
| **Hydration** | A claim is hydrated when it respects its constraint basis (angle < 10°) |

### Connection to Your PCSP Seminar

The speaker's actual claim — *"Some dichotomy results for Mal'cev algebras; general case open"* — is **hydrated (VC)** because:
- It anchors to known CSP dichotomy (C1 ✓)
- It specifies the algebra class (C2 ✓)
- The promise condition is central (C3 ✓)
- The talk presents actual constructions (C4 ✓)
- It acknowledges the open general case (C5 ✓)

The **invalid assignment** would be claiming the general dichotomy is solved — a 45° claim that collapses under the cosine constraint.

### Next Steps

1. **Apply the framework** to your own research claims
2. **Use the notebooks** as diagnostic tools
3. **Extend the repository** with new domains
4. **Share with colleagues** — the 45° boundary appears everywhere

---

**Thank you for using the Cosine Constraint Lab!**

Remember: **Construction ≠ Assignment. Stay locked below 10°.** 🔒